# MuseumSCAT Specimen Label Baseline

Fine-tunes Qwen2.5-VL-7B (4-bit, LoRA) on 200 museum specimen label images to extract verbatim collection dates and localities from Danish dung beetle specimens

Necessary imports

In [1]:
#!pip install -U -q bitsandbytes transformers accelerate qwen-vl-utils[decord]==0.0.8 matplotlib unsloth peft trl
# !rm -rf /kaggle/working/*

## imports and paths

In [2]:
import os
import torch
# ─── ADD THIS HERE (ABSOLUTE TOP OF EXECUTION) ───
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

if torch.cuda.is_available():
    # Force a quick, physical hardware initialization handshake
    _ = torch.tensor([1.0]).cuda()
    print(f"CUDA initialized successfully on: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is not available to PyTorch!")
# ────────────────────────────────────────────────
#from unsloth import FastVisionModel
#from unsloth.trainer import UnslothVisionDataCollator
#from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor
from transformers import AutoProcessor, AutoModelForImageTextToText#AutoModelForVision2Seq
from huggingface_hub import snapshot_download
from qwen_vl_utils import process_vision_info
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import json
import shutil
from transformers import Qwen2_5_VLProcessor
from trl import SFTConfig
import re
import gc

# ── Toggle ───────────────────────────────────────────────────────────────────
SMOKE      = False
BATCH_SIZE = 10 # inference
T_BATCH_SIZE = 1
TRAIN_EPOCHS = 5
# ─────────────────────────────────────────────────────────────────────────────

images_dir = "/home/soffer/kaggle/MuseumSCAT/museumscat-specimen-collection-annotation-task/images"
test_file  = "/home/soffer/kaggle/MuseumSCAT/museumscat-specimen-collection-annotation-task/test.csv"
train_file = "/home/soffer/kaggle/MuseumSCAT/museumscat-specimen-collection-annotation-task/train.csv"
model_path_3b = "/home/soffer/kaggle/MuseumSCAT/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct"
dest_dir   = "/home/soffer/kaggle/MuseumSCAT/working/output"
train_df   = pd.read_csv(train_file)
test_df    = pd.read_csv(test_file)
print("train df"); print(train_df.head())
print("test_df");  print(test_df.head())



CUDA initialized successfully on: NVIDIA GeForce RTX 3050 6GB Laptop GPU
train df
     image_file verbatimDate  verbatimDate_confidence verbatimLocality  \
0  1709018.jpeg      MISSING                        1          MISSING   
1  1709789.jpeg    22.5.1977                        1     Svinø strand   
2  1706390.jpeg     1.7.2000                        1       Lodskovvad   
3  1709042.jpeg      MISSING                        1          MISSING   
4  1707822.jpeg      MISSING                        1               Ti   

   verbatimLocality_confidence  
0                            1  
1                            1  
2                            1  
3                            1  
4                            1  
test_df
     image_file
0  1709398.jpeg
1  1705937.jpeg
2  1705822.jpeg
3  1742998.jpeg
4  1728459.jpeg


## one time operations: download foundary model, crop and resize images 

In [3]:


# # 1. Download the raw files directly into your target directory
# from huggingface_hub import snapshot_download

# # Define a clean path for the 3B model variant

# print("Downloading Qwen2.5-VL-3B-Instruct directly to local workspace...")
# snapshot_download(
#     repo_id="Qwen/Qwen2.5-VL-3B-Instruct", 
#     local_dir=model_path_3b,
#     local_dir_use_symlinks=False
# )
# print("3B files ready!")


# def crop_and_resize_img(image_path, dest_dir):
#     try:
#         img = Image.open(image_path)
#         width, height = img.size
#         box = (
#             int(0.25 * width),
#             int(0.1  * height),
#             int(0.95 * width),
#             int(0.41 * height)
#         )
#         cropped_img = img.crop(box)
#         new_width   = 512
#         crop_width, crop_height = cropped_img.size
#         new_height  = int((new_width / crop_width) * crop_height)
#         resized_img = cropped_img.resize((new_width, new_height))
#         save_path   = os.path.join(dest_dir, os.path.basename(image_path))
#         resized_img.save(save_path)
#     except Exception as e:
#         print(f"Error processing {image_path}: {e}")

# source_dir = images_dir
# os.makedirs(dest_dir, exist_ok=True)



# image_paths = [
#     os.path.join(source_dir, f)
#     for f in os.listdir(source_dir)
#     if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
# ]

# num_workers = os.cpu_count()
# print("using num of workers:", num_workers)
# with ProcessPoolExecutor(max_workers=num_workers) as executor:
#     list(tqdm(
#         executor.map(crop_and_resize_img, image_paths, [dest_dir] * len(image_paths)),
#         total=len(image_paths)
#     ))


## define promt

In [4]:

propmt = """
You are an expert museum specimen label transcription system.
Carefully read the date and location of the image and extract the requested metadata.

The image contains multiple label cards. You must:
- READ: The small handwritten or printed card showing the collection DATE and LOCATION/LOCALITY.
- IGNORE: Any card showing species names (e.g. "A. subte raneus"), collector names (e.g. "Coll. Rosenberg"), museum logos, barcodes, or color calibration cards.
- The date and locality are typically on the SAME small card together.
- If a card says "Coll." or "det." or "Tilg." followed by a name or date, ignore that card entirely.
- If a card contains a latin species name, ignore it.
- "Dania" is NOT a locality — it refers to the museum collection. Output "MISSING" for locality if only "Dania" appears.
- Phrases meaning "in cow dung" (e.g. "i kogøding", "i kogjorning") are NOT locality — ignore them.
- If there are MULTIPLE label cards with date/locality info, transcribe ALL of them separated by " | " (pipe). Example: verbatimLocality: "Ti | Kulhuset Jægerspris", verbatimDate: "6/1862"
- Even if date/locality repeats across cards, transcribe both: "10.6.1951 | 10.6.1951"

Rules:
1. Return ONLY valid JSON.
2. Do not include markdown.
3. Do not explain your reasoning.
4. Preserve original spelling EXACTLY as written — do not correct, expand, or normalize anything.
5. If a field is unreadable or missing, output "MISSING".
6. Confidence must be a decimal number between 0 and 1.0.
7. 0.0 confidence means completely uncertain while 1.0 means completely certain in answer.
8. Higher confidence means higher certainty from visual evidence.
9. Never hallucinate values not visible in the image.
10. Preserve Scandinavian and special Nordic characters exactly when visible (æ, ø, å, ä, ö). Use standard English characters otherwise.
11. For dates: preserve the EXACT format as written. Do NOT expand abbreviations (e.g. "Septmbr" stays "Septmbr", not "September"). Do NOT convert Roman numerals (e.g. "IV" stays "IV"). Do NOT reformat or normalize dates.
12. For locality: preserve EXACTLY as written including abbreviations (e.g. "Kb" stays "Kb", not "København").
13. Use low confidence (< 0.5) when: handwriting is faded, ink is smeared, text is partially obscured, or you can only partially read a word.
14. Use medium confidence (0.5-0.8) when: handwriting is clear but unusual style, or text is fully readable but abbreviated/ambiguous.
15. Use high confidence (0.9-1.0) when: text is clearly printed or very legible handwriting with no ambiguity.

Required JSON schema:
{
  "verbatimDate": "string",
  "verbatimDate_confidence": float,
  "verbatimLocality": "string",
  "verbatimLocality_confidence": float
}

Examples:
{"verbatimDate": "22.5.1977", "verbatimDate_confidence": 0.98, "verbatimLocality": "Svinø strand", "verbatimLocality_confidence": 0.95}
{"verbatimDate": "MISSING", "verbatimDate_confidence": 1.0, "verbatimLocality": "MISSING", "verbatimLocality_confidence": 1.0}
{"verbatimDate": "22 VIII 2027", "verbatimDate_confidence": 1.0, "verbatimLocality": "Evæglion", "verbatimLocality_confidence": 1.0}
{"verbatimDate": "18/7 70", "verbatimDate_confidence": 0.95, "verbatimLocality": "Faaborg", "verbatimLocality_confidence": 0.95}
{"verbatimDate": "Septmbr 1923", "verbatimDate_confidence": 0.95, "verbatimLocality": "Kb", "verbatimLocality_confidence": 1.0}
{"verbatimDate": "10/5 14", "verbatimDate_confidence": 0.90, "verbatimLocality": "Grønne Vestkile", "verbatimLocality_confidence": 0.98}
{"verbatimDate": "Maj 1897", "verbatimDate_confidence": 0.45, "verbatimLocality": "Vordingbg", "verbatimLocality_confidence": 0.40}
{"verbatimDate": "6/1862", "verbatimDate_confidence": 0.95, "verbatimLocality": "Ti | Kulhuset Jægerspris", "verbatimLocality_confidence": 0.90}
{"verbatimDate": "10.6.1951 | 10.6.1951", "verbatimDate_confidence": 0.98, "verbatimLocality": "Rotholme Jyll. | Rotholme Jyll.", "verbatimLocality_confidence": 0.98}
"""


## model generation and training

In [5]:

# ── Patch preprocessor_config.json ──────────────────────────────────────────
writable_model_path = "/home/soffer/kaggle/working/model_patched"
os.makedirs(writable_model_path, exist_ok=True)

for f in os.listdir(model_path_3b):
    src = os.path.realpath(os.path.join(model_path_3b, f))
    dst = os.path.join(writable_model_path, f)
    if os.path.exists(dst) or os.path.islink(dst):
        os.remove(dst)
    if f.endswith((".json", ".txt", ".py", ".md")):
        shutil.copy2(src, dst)
    else:
        os.symlink(src, dst)

cfg_path = os.path.join(writable_model_path, "preprocessor_config.json")
with open(cfg_path, "r") as f:
    cfg = json.load(f)
cfg["image_processor_type"] = "Qwen2VLImageProcessor"
with open(cfg_path, "w") as f:
    json.dump(cfg, f)



import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. CRITICAL STEP: Define strict resolution boundaries to limit token explosion
# This restricts the image tokens to a tight, memory-friendly sequence range
min_pixels = 256 * 28 * 28   # Lower bound
max_pixels = 512 * 28 * 28   # Upper bound (Keeps VRAM footprint tiny!)


# Load the processor with the pixel constraints
processor = AutoProcessor.from_pretrained(
    model_path_3b, 
    min_pixels=min_pixels, 
    max_pixels=max_pixels
)

torch.cuda.empty_cache()

# 1. Setup the exact 4-bit quantization config (replaces load_in_4bit=True)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading 3B model in 4-bit precision...")
# 3. Load with explicit memory-streaming configurations
model = AutoModelForImageTextToText.from_pretrained(
    model_path_3b,
    quantization_config=bnb_config,
    device_map="auto",              # CRITICAL: Let Hugging Face balance it dynamically
    low_cpu_mem_usage=True,         # CRITICAL: Do not allocate raw model buffers on GPU
    torch_dtype=torch.bfloat16      # Keeps operations in half-precision bounds
)


model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

peft_config = LoraConfig(
    r=8, 
    lora_alpha=8,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0, 
    bias="none",
    task_type="CAUSAL_LM"
)

print("Applying LoRA adapters...")
model = get_peft_model(model, peft_config)
processor = AutoProcessor.from_pretrained(model_path_3b)

print("3B Model and processor successfully loaded! Your GPU can comfortably handle this workflow.")


# 6. Load the processor natively (Qwen2_5_VLProcessor maps directly to AutoProcessor)
processor = AutoProcessor.from_pretrained(writable_model_path)

print("Model and processor successfully initialized natively!")

def make_row(row):
    img_path  = os.path.join(dest_dir, row["image_file"])
    date_val  = str(row["verbatimDate"])
    loc_val   = str(row["verbatimLocality"])
    # Training confidences are all 1.0 (ground truth artifact) — use realistic values instead
    date_conf = 1.0 if date_val == "MISSING" else 0.95
    loc_conf  = 1.0 if loc_val  == "MISSING" else 0.95
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": img_path},
                {"type": "text",  "text": propmt},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": json.dumps({
                    "verbatimDate":                date_val,
                    "verbatimDate_confidence":     date_conf,
                    "verbatimLocality":            loc_val,
                    "verbatimLocality_confidence": loc_conf,
                })}
            ]},
        ]
    }

src        = train_df.head(20) if SMOKE else train_df
hf_dataset = Dataset.from_list([make_row(r) for _, r in src.iterrows()])
print(f"Training on {len(hf_dataset)} samples (smoke={SMOKE})")
# #
# # Run this in a new cell to inspect the training format
# first_sample = hf_dataset[0]
# #print("--- DATASET KEYS ---")
# print(0, first_sample['messages'][0]['content'][0]['text'])
# print(1, first_sample['messages'][0]['content'][1]['text'])
# assert False
# #

# ── Fine-tune ────────────────────────────────────────────────────────────────
# 1. Comment out the Unsloth wrapper call
# FastVisionModel.for_training(model)

# # 2. Feed the native model and processor into the trainer
# trainer = SFTTrainer(
#     model=model,
#     processing_class=processor,  # Replaces tokenizer=tokenizer for VLMs
#     train_dataset=hf_dataset,
#     # ... keep the rest of the author's original trainer arguments here (args, dataset_text_field, etc.) ...
# )

#trainer.train()

if 'trainer' in locals(): 
    del trainer
gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    processing_class=processor,
    train_dataset=hf_dataset,
    args=SFTConfig(
        output_dir="/home/soffer/kaggle/MuseumSCAT/checkpoints",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=20,
        optim="adamw_8bit",
        num_train_epochs=TRAIN_EPOCHS,
        learning_rate=2e-4,
        warmup_steps=100,
        lr_scheduler_type="cosine",
        fp16=False, # <── This explicit flag overrides the broken checker!
        bf16=True,  # <── This explicit flag overrides the broken checker!
        logging_steps=5,
        save_strategy="no",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": False},
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
    ),
)

if TRAIN_EPOCHS > 0:
    trainer.train()
    model.save_pretrained("/home/soffer/kaggle/MuseumSCAT/working/lora_adapter")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading 3B model in 4-bit precision...


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Applying LoRA adapters...
3B Model and processor successfully loaded! Your GPU can comfortably handle this workflow.
Model and processor successfully initialized natively!
Training on 200 samples (smoke=False)


Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


KeyboardInterrupt: 

## inference

In [6]:

# ── Raw output ─────────────────────────────────────────────────────────────
#FastVisionModel.for_inference(model)
model.eval()

test_row    = test_df.iloc[0]
img_path    = os.path.join(dest_dir, test_row["image_file"])
msgs        = [{"role": "user", "content": [
    {"type": "image", "image": img_path},
    {"type": "text",  "text": propmt},
]}]
text        = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
img_info, _ = process_vision_info(msgs)
inputs = processor(text=[text], images=img_info, return_tensors="pt").to(model.device)
if "pixel_values" in inputs:
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    
with torch.no_grad():
    with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)

#raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
raw = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== RAW OUTPUT ==="); print(raw); print("==================")

# ── Batch inference ──────────────────────────────────────────────────────────
def clean_json(text):
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*",     "", text)
    return text.strip()

# import gc
# import torch

def run_inference(df, prompt):
    results = []
    
    # Force evaluation mode
    model.eval()
    
    print(f"Starting batch-by-batch inference on {len(df)} samples...")
    
    # Process exactly 1 row at a time to minimize memory footprints
    for idx, row in df.iterrows():
        # if idx > 25:
        #     continue   # for quick debug

        # Clear VRAM cache aggressively between rows
        gc.collect()
        torch.cuda.empty_cache()

        #text = row["text"] # update to match your prompt string column name if different
        img_file = os.path.join(dest_dir, row["image_file"])
        
        # 1. Load image safely
        from PIL import Image
        try:
            img_info = Image.open(img_file).convert("RGB")
        except Exception:
            img_info = None
            
        # ─── THE FIX: Format the prompt string for multi-modal inference ───
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img_file},
                    {"type": "text", "text": propmt}
                ]
            }
        ]
        # This inserts the critical <|image_pad|> tags Qwen needs to align the features
        formatted_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)        # 2. Process input with an explicit max token pixel boundary

        inputs = processor(
            text=[formatted_text],
            images=[img_info] if img_info else None,
            return_tensors="pt",
        ).to(model.device)
        
        # Cast vision tensors to half precision
        if "pixel_values" in inputs:
            inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
            
        # 3. Generate with strict constraints
        with torch.no_grad():
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=256, # Tightened to save generation step VRAM
                    do_sample=False,
                    pad_token_id=processor.tokenizer.eos_token_id,
                    use_cache=True     # Speeds up sequential token generation memory reuse
                )
                
        # 4. Decode text
        prompt_len = inputs["input_ids"].shape[1]
        decoded_text = processor.decode(out_ids[0][prompt_len:], skip_special_tokens=True)
        
        # Append target key format to list (assuming standard sample submission headers)
        results.append({
            "image_file": row["image_file"],
            "prediction": decoded_text.strip()
        })
        
        if (idx + 1) % 5 == 0:
            print(f"Processed {idx + 1}/{len(df)} rows successfully.")
            
    return pd.DataFrame(results)

# ── Execution Block ─────────────────────────────────────────────────────────
# infer_df = test_df.head(5) if SMOKE else test_df
# preds_df = run_inference(infer_df)
# preds_df.to_csv("/home/soffer/kaggle/MuseumSCAT/working/submission_pre.csv", index=False)
# print(preds_df)
# Clear out lingering baseline training states first
gc.collect()
torch.cuda.empty_cache()

infer_df = test_df.head(5) if SMOKE else test_df
preds_df = run_inference(infer_df, propmt)
preds_df.to_csv("/home/soffer/kaggle/MuseumSCAT/working/submission_pre.csv", index=False)
print("Submission saved cleanly!")
print(preds_df.head())


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== RAW OUTPUT ===
```json
{
  "verbatimDate": "12.5.2022",
  "verbatimDate_confidence": 0.95,
  "verbatimLocality": "Helsingør, Grønne Vestkile, 56.0375N, 12.5598E",
  "verbatimLocality_confidence": 0.95
}
```
Starting batch-by-batch inference on 3300 samples...
Processed 5/3300 rows successfully.
Processed 10/3300 rows successfully.
Processed 15/3300 rows successfully.
Processed 20/3300 rows successfully.
Processed 25/3300 rows successfully.
Processed 30/3300 rows successfully.
Processed 35/3300 rows successfully.
Processed 40/3300 rows successfully.
Processed 45/3300 rows successfully.
Processed 50/3300 rows successfully.
Processed 55/3300 rows successfully.
Processed 60/3300 rows successfully.
Processed 65/3300 rows successfully.
Processed 70/3300 rows successfully.
Processed 75/3300 rows successfully.
Processed 80/3300 rows successfully.
Processed 85/3300 rows successfully.
Processed 90/3300 rows successfully.
Processed 95/3300 rows successfully.
Processed 100/3300 rows successful

## post-process

In [8]:


# def postprocess(text):
#     if not isinstance(text, str) or text == "MISSING":
#         return "MISSING"
#     for old, new in {"ö":"ø","Ö":"Ø","ä":"æ","Ä":"Æ","ü":"y","Ü":"Y","ÿ":"y","Ÿ":"Y"}.items():
#         text = text.replace(old, new)
#     return text

# print(preds_df.iloc[0]['prediction'])
# preds_df['prediction']["verbatimDate"]     = preds_df['prediction']["verbatimDate"].apply(postprocess)
# preds_df['prediction']["verbatimLocality"] = preds_df['prediction']["verbatimLocality"].apply(postprocess)
# preds_df.to_csv("/home/soffer/kaggle/MuseumSCAT/working/submission.csv", index=False)
# print(preds_df.head(10))

import json
import pandas as pd
import re

def postprocess(text):
    if not isinstance(text, str) or text == "MISSING":
        return "MISSING"
    for old, new in {"ö":"ø","Ö":"Ø","ä":"æ","Ä":"Æ","ü":"y","Ü":"Y","ÿ":"y","Ÿ":"Y"}.items():
        text = text.replace(old, new)
    return text

parsed_rows = []

for idx, row in preds_df.iterrows():
    raw_text = row["prediction"].strip()
    
    # 1. Clean out markdown block wrappers if present
    raw_text = re.sub(r"```json\s*|\s*```", "", raw_text).strip()
    
    parsed_json = None
    
    # Strategy A: Attempt standard clean load
    try:
        parsed_json = json.loads(raw_text)
    except Exception:
        # Strategy B: If it fails due to trailing text, extract just the first {...} block
        try:
            match = re.search(r"(\{.*?\})", raw_text, re.DOTALL)
            if match:
                parsed_json = json.loads(match.group(1))
        except Exception as e:
            print(f"Warning: Complete structural parse failure at index {idx}: {e}")

    # Strategy C: If it's still missing, or if Strategy A returned a list instead of a dict
    if isinstance(parsed_json, list) and len(parsed_json) > 0:
        parsed_json = parsed_json[0] # Grab the dictionary inside the list
        
    if not isinstance(parsed_json, dict):
        # Fallback to defaults if everything failed
        parsed_json = {
            "verbatimDate": "MISSING",
            "verbatimDate_confidence": 1.0,
            "verbatimLocality": "MISSING",
            "verbatimLocality_confidence": 1.0
        }

    # Build the safe dictionary row mapping
    parsed_rows.append({
        "image_file": row["image_file"],
        "verbatimDate": parsed_json.get("verbatimDate", "MISSING"),
        "verbatimDate_confidence": parsed_json.get("verbatimDate_confidence", 1.0),
        "verbatimLocality": parsed_json.get("verbatimLocality", "MISSING"),
        "verbatimLocality_confidence": parsed_json.get("verbatimLocality_confidence", 1.0)
    })

# Overwrite preds_df with the structural multi-column layout
preds_df = pd.DataFrame(parsed_rows)

# ── 2. Run the baseline post-processing scripts ─────────────────────────────
print("--- Raw value sample before postprocessing ---")
if not preds_df.empty:
    print(preds_df.iloc[0]['verbatimLocality'])

preds_df["verbatimDate"]     = preds_df["verbatimDate"].apply(postprocess)
preds_df["verbatimLocality"] = preds_df["verbatimLocality"].apply(postprocess)

# ── 3. Save to clean submission format ──────────────────────────────────────
preds_df.to_csv("/home/soffer/kaggle/MuseumSCAT/working/submission.csv", index=False)

print("\n--- Final Submission Preview ---")
print(preds_df.head(10))

--- Raw value sample before postprocessing ---
Helsingør, Grønne Vestkile, 56.0375N, 12.5598E

--- Final Submission Preview ---
     image_file verbatimDate  verbatimDate_confidence  \
0  1709398.jpeg    12.5.2022                     0.95   
1  1705937.jpeg    18/7 1965                     0.95   
2  1705822.jpeg      MISSING                     0.95   
3  1742998.jpeg    21-5-1951                     0.95   
4  1728459.jpeg      MISSING                     0.95   
5  1707402.jpeg      MISSING                     0.00   
6  1707293.jpeg    10.6.1951                     0.98   
7  1744483.jpeg      MISSING                     0.95   
8  1708366.jpeg     9.5.1944                     0.95   
9  1705671.jpeg      MISSING                     0.95   

                                 verbatimLocality  verbatimLocality_confidence  
0  Helsingør, Grønne Vestkile, 56.0375N, 12.5598E                         0.95  
1                                Dania V.Sjælland                         0.95  
2